# Singapore Criminal Law Dataset — Exploratory Data Analysis
## 876 Supreme Court Judgments Analysis

This notebook provides comprehensive exploratory analysis of the criminal law dataset, including distribution patterns, categorical breakdowns, and data quality insights.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('dataset.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (1683, 11)

First few rows:


,filename,case_name,citation,catchword,area_of_law,topic,subtopic,primary_statute,is_criminal,taxonomy_key,pdf_url
0,2026_SGCA_15.pdf,[2026] SGCA 15,[2026] SGCA 15,Criminal Law — Statutory offences — Misuse of ...,Criminal Law,Statutory offences,Misuse of Drugs Act,Misuse of Drugs Act,True,Criminal Law — Statutory offences — Misuse of ...,https://www.elitigation.sg/gd/gd/2026_SGCA_15/pdf
1,2026_SGCA_15.pdf,[2026] SGCA 15,[2026] SGCA 15,Criminal Procedure and Sentencing — Appeal — A...,Criminal Procedure,General,General,Criminal Procedure Code 2010,True,Criminal Procedure and Sentencing,https://www.elitigation.sg/gd/gd/2026_SGCA_15/pdf
2,2026_SGCA_15.pdf,[2026] SGCA 15,[2026] SGCA 15,Abuse of process — Inconsistent positions — Ap...,Criminal Procedure,Abuse of process,General,Criminal Procedure Code 2010,True,Abuse of process,https://www.elitigation.sg/gd/gd/2026_SGCA_15/pdf
3,2026_SGCA_15.pdf,[2026] SGCA 15,[2026] SGCA 15,Abuse of process — Collateral purpose — Appell...,Criminal Procedure,Abuse of process,General,Criminal Procedure Code 2010,True,Abuse of process,https://www.elitigation.sg/gd/gd/2026_SGCA_15/pdf
4,2026_SGCA_13.pdf,[2026] SGCA 13,[2026] SGCA 13,Criminal Procedure and Sentencing — Sentencing...,Criminal Law,Sentencing,General,Criminal Procedure Code 2010,True,Criminal Procedure and Sentencing — Sentencing,https://www.elitigation.sg/gd/gd/2026_SGCA_13/pdf


## 1. Dataset Overview & Basic Statistics

In [2]:
# Basic statistics
unique_files = df['filename'].nunique()
unique_cases = df['case_name'].nunique()
unique_areas = df['area_of_law'].nunique()
unique_topics = df['topic'].nunique()
unique_subtopics = df['subtopic'].nunique()
unique_statutes = df['primary_statute'].nunique()

stats_df = pd.DataFrame({
    'Metric': ['Total Rows', 'Unique Case Files', 'Unique Cases', 'Areas of Law', 'Topics', 'Subtopics', 'Primary Statutes'],
    'Count': [len(df), unique_files, unique_cases, unique_areas, unique_topics, unique_subtopics, unique_statutes]
})

fig = go.Figure(data=[
    go.Bar(x=stats_df['Metric'], y=stats_df['Count'], text=stats_df['Count'], textposition='auto', marker_color='#1f77b4')
])
fig.update_layout(title="Dataset Overview Statistics", yaxis_title="Count", xaxis_tickangle=-45, height=400)
fig.show()

print(stats_df.to_string(index=False))

           Metric  Count
       Total Rows   1683
Unique Case Files    876
     Unique Cases    876
     Areas of Law     22
           Topics     79
        Subtopics    166
 Primary Statutes    114


In [26]:
# Check data types and missing values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nNo missing values: {df.isnull().sum().sum() == 0}")

Missing Values:
filename                 0
case_name                0
citation                 0
catchword                0
area_of_law              0
topic                    1
subtopic                12
primary_statute         34
is_criminal              0
taxonomy_key            34
pdf_url                  0
catchword_length         0
catchword_word_count     0
court_level              3
year                     0
topic_subtopic          12
dtype: int64

No missing values: False


## 2. Area of Law Distribution

In [3]:
# Distribution by Area of Law
area_counts = df['area_of_law'].value_counts().head(15)

fig = go.Figure(data=[
    go.Bar(x=area_counts.values, y=area_counts.index, orientation='h', marker_color='#2ca02c')
])
fig.update_layout(
    title="Top 15 Areas of Law (by row count)",
    xaxis_title="Count",
    yaxis_title="Area of Law",
    height=500,
    margin=dict(l=300)
)
fig.show()

print(f"\nTotal areas of law: {df['area_of_law'].nunique()}")
print(f"\nTop 10 areas:")
print(df['area_of_law'].value_counts().head(10))


Total areas of law: 22

Top 10 areas:
area_of_law
Criminal Law                        1148
Criminal Procedure                   393
Constitutional Law                    42
Evidence                              34
Statutory Interpretation              26
Administrative Law                    10
Civil Procedure                        7
Agency                                 5
Financial and Securities Markets       3
Legal Profession                       2
Name: count, dtype: int64


## 3. Topics & Subtopics Breakdown

In [12]:
# Topics Distribution
topic_counts = df['topic'].value_counts().head(12)

fig = px.bar(
    x=topic_counts.values, 
    y=topic_counts.index,
    orientation='h',
    color=topic_counts.values,
    color_continuous_scale='Viridis',
    labels={'x': 'Count', 'y': 'Topic'}
)
fig.update_layout(
    title="Top 12 Topics Distribution",
    height=500,
    margin=dict(l=250),
    showlegend=False
)
fig.show()

print(f"Total unique topics: {df['topic'].nunique()}")
print(f"\nTop topics:")
print(df['topic'].value_counts().head(10))

Total unique topics: 79

Top topics:
topic
Sentencing                       446
Statutory offences               366
General                          208
Sexual offences                  127
Offences against person           72
General exceptions / Defences     34
Review                            33
Property offences                 32
Appeals                           30
Statements                        29
Name: count, dtype: int64


In [13]:
# Most common Topic-Subtopic combinations
df['topic_subtopic'] = df['topic'] + ' — ' + df['subtopic']
topic_subtopic_counts = df['topic_subtopic'].value_counts().head(15)

fig = go.Figure(data=[
    go.Bar(x=topic_subtopic_counts.values, y=topic_subtopic_counts.index, orientation='h', 
           marker_color='#ff7f0e')
])
fig.update_layout(
    title="Top 15 Topic-Subtopic Combinations",
    xaxis_title="Count",
    yaxis_title="Topic — Subtopic",
    height=600,
    margin=dict(l=400)
)
fig.show()

## 4. Primary Statute Analysis

In [14]:
# Distribution of Primary Statutes
statute_counts = df['primary_statute'].value_counts().head(15)

fig = px.pie(
    values=statute_counts.values,
    names=statute_counts.index,
    title="Top 15 Primary Statutes (Pie Chart)",
    hole=0.3
)
fig.update_layout(height=600)
fig.show()

print(f"Total unique primary statutes: {df['primary_statute'].nunique()}")
print(f"\nTop 15 statutes:")
print(statute_counts)

Total unique primary statutes: 114

Top 15 statutes:
primary_statute
Criminal Procedure Code 2010           697
Misuse of Drugs Act                    220
Penal Code                             109
Penal Code s 375                        46
Penal Code Pt XI                        41
Criminal Procedure Code 2010 s 394H     33
Evidence Act                            31
Penal Code s 300                        28
Criminal Procedure Code 2010 s 397      27
Interpretation Act                      26
Penal Code s 354                        20
Constitution of Singapore Art 12        14
Penal Code s 299                        14
Supreme Court of Judicature Act         14
Prevention of Corruption Act            14
Name: count, dtype: int64


In [15]:
# Statute bar chart for easier comparison
fig = go.Figure(data=[
    go.Bar(x=statute_counts.values, y=statute_counts.index, orientation='h',
           marker=dict(color=statute_counts.values, colorscale='Plasma', showscale=True))
])
fig.update_layout(
    title="Top 15 Primary Statutes (Bar Chart)",
    xaxis_title="Count",
    height=600,
    margin=dict(l=300)
)
fig.show()

## 5. Catchword Patterns & Text Analysis

In [16]:
# Catchword length analysis
df['catchword_length'] = df['catchword'].str.len()
df['catchword_word_count'] = df['catchword'].str.split().str.len()

fig = go.Figure()
fig.add_trace(go.Histogram(x=df['catchword_length'], name='Character Length', nbinsx=50, opacity=0.7))
fig.update_layout(
    title="Distribution of Catchword Length (Characters)",
    xaxis_title="Character Count",
    yaxis_title="Frequency",
    height=400
)
fig.show()

print(f"Catchword length statistics:")
print(df['catchword_length'].describe())
print(f"\nCatchword word count statistics:")
print(df['catchword_word_count'].describe())

Catchword length statistics:
count    1683.000000
mean       63.506239
std        33.034165
min        10.000000
25%        46.000000
50%        55.000000
75%        68.000000
max       489.000000
Name: catchword_length, dtype: float64

Catchword word count statistics:
count    1683.000000
mean        9.810458
std         5.239027
min         1.000000
25%         7.000000
50%         9.000000
75%        10.000000
max        83.000000
Name: catchword_word_count, dtype: float64


In [17]:
# Extract key terms from catchwords (words appearing frequently)
from collections import Counter

# Extract all words from catchwords (filter common words)
stop_words = {'the', 'a', 'an', 'and', 'or', 'of', 'in', 'to', 'is', 'was', 'are'}
all_words = []

for catchword in df['catchword']:
    words = [w.lower().strip('—').strip() for w in str(catchword).split()]
    words = [w for w in words if w and w not in stop_words and len(w) > 2]
    all_words.extend(words)

word_freq = Counter(all_words).most_common(20)
words, counts = zip(*word_freq)

fig = go.Figure(data=[
    go.Bar(x=counts, y=words, orientation='h', marker_color='#d62728')
])
fig.update_layout(
    title="Top 20 Most Frequent Words in Catchwords",
    xaxis_title="Frequency",
    height=500,
    margin=dict(l=200)
)
fig.show()

## 6. Case Citation Patterns

In [18]:
# Extract court level from citation (SGCA, SGHC, etc.)
df['court_level'] = df['citation'].str.extract(r'(SGCA|SGHC|SGDC|SGPA)')

court_counts = df['court_level'].value_counts()

fig = go.Figure(data=[
    go.Bar(x=court_counts.index, y=court_counts.values, marker_color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(court_counts)])
])
fig.update_layout(
    title="Distribution by Court Level",
    xaxis_title="Court",
    yaxis_title="Count",
    height=400
)
fig.show()

print(f"\nCourt distribution:")
print(court_counts)
print(f"\nPercentage:")
print((court_counts / len(df) * 100).round(2))


Court distribution:
court_level
SGHC    1267
SGCA     413
Name: count, dtype: int64

Percentage:
court_level
SGHC    75.28
SGCA    24.54
Name: count, dtype: float64


In [19]:
# Extract year from citation
df['year'] = df['citation'].str.extract(r'(\d{4})', expand=False)
df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(2026).astype(int)

year_counts = df['year'].value_counts().sort_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=year_counts.index, y=year_counts.values, mode='lines+markers',
                         name='Cases', line=dict(color='#1f77b4', width=2), marker=dict(size=8)))
fig.update_layout(
    title="Cases Over Time",
    xaxis_title="Year",
    yaxis_title="Number of Cases",
    height=400,
    hovermode='x unified'
)
fig.show()

print(f"\nCases by year:")
print(year_counts.tail(10))


Cases by year:
year
2017    141
2018    119
2019    165
2020    168
2021    187
2022    167
2023    203
2024    187
2025    174
2026     69
Name: count, dtype: int64


## 7. Cross-tabulations & Relationships

In [20]:
# Area of Law vs Court Level
area_court = pd.crosstab(df['area_of_law'], df['court_level'])
top_areas = df['area_of_law'].value_counts().head(8).index
area_court_subset = area_court.loc[top_areas]

fig = go.Figure()
for court in area_court_subset.columns:
    fig.add_trace(go.Bar(name=court, x=area_court_subset.index, y=area_court_subset[court]))

fig.update_layout(
    title="Top 8 Areas of Law by Court Level",
    barmode='stack',
    xaxis_tickangle=-45,
    height=500,
    margin=dict(b=150)
)
fig.show()

In [21]:
# Area of Law vs Year (recent years only)
recent_years = [2024, 2025, 2026]
df_recent = df[df['year'].isin(recent_years)]

area_year = pd.crosstab(df_recent['area_of_law'], df_recent['year'])
top_areas_recent = df_recent['area_of_law'].value_counts().head(10).index
area_year_subset = area_year.loc[top_areas_recent]

fig = go.Figure()
for year in sorted(area_year_subset.columns):
    fig.add_trace(go.Bar(name=str(year), x=area_year_subset.index, y=area_year_subset[year]))

fig.update_layout(
    title="Top 10 Areas of Law by Recent Years (2024-2026)",
    barmode='group',
    xaxis_tickangle=-45,
    height=500,
    margin=dict(b=150)
)
fig.show()

## 8. Data Quality & Coverage Assessment

In [22]:
# Cases per unique filename (how many rows per case file)
rows_per_file = df.groupby('filename').size().value_counts().sort_index()

fig = go.Figure()
fig.add_trace(go.Bar(x=rows_per_file.index, y=rows_per_file.values, marker_color='#9467bd'))
fig.update_layout(
    title="Distribution: Rows per Case File",
    xaxis_title="Number of Rows (Catchwords/Topics)",
    yaxis_title="Number of Case Files",
    height=400
)
fig.show()

print(f"Average rows per case file: {len(df) / df['filename'].nunique():.2f}")
print(f"\nDistribution of rows per file:")
print(rows_per_file)

Average rows per case file: 1.92

Distribution of rows per file:
1     386
2     295
3     131
4      38
5       9
6      12
8       3
10      1
11      1
Name: count, dtype: int64


In [23]:
# Coverage by taxonomy_key
taxonomy_coverage = df['taxonomy_key'].value_counts().head(15)

fig = go.Figure(data=[
    go.Bar(x=taxonomy_coverage.values, y=taxonomy_coverage.index, orientation='h',
           marker=dict(color=taxonomy_coverage.values, colorscale='RdYlGn', showscale=True))
])
fig.update_layout(
    title="Top 15 Taxonomy Keys (by coverage)",
    xaxis_title="Count",
    height=600,
    margin=dict(l=350)
)
fig.show()

print(f"Total unique taxonomy keys: {df['taxonomy_key'].nunique()}")

Total unique taxonomy keys: 175


In [24]:
# Validation: all rows should have is_criminal = True
print("Data Quality Checks:")
print(f"✓ All rows have is_criminal = True: {(df['is_criminal'] == True).all() or (df['is_criminal'] == 'True').all()}")
print(f"✓ No missing filenames: {df['filename'].isnull().sum() == 0}")
print(f"✓ No missing citations: {df['citation'].isnull().sum() == 0}")
print(f"✓ No missing areas of law: {df['area_of_law'].isnull().sum() == 0}")
print(f"✓ No missing topics: {df['topic'].isnull().sum() == 0}")
print(f"✓ All PDF URLs valid format: {df['pdf_url'].str.startswith('https://').all()}")
print(f"\nTotal rows with complete data: {len(df)}")

Data Quality Checks:
✓ All rows have is_criminal = True: False
✓ No missing filenames: True
✓ No missing citations: True
✓ No missing areas of law: True
✓ No missing topics: False
✓ All PDF URLs valid format: True

Total rows with complete data: 1683


## 9. Summary Insights

In [25]:
summary_insights = f"""
📊 SINGAPORE CRIMINAL LAW DATASET — KEY INSIGHTS
================================================

📁 DATASET COMPOSITION:
  • Total data points: {len(df):,} rows (multiple catchwords per case)
  • Unique cases: {unique_cases:,} criminal judgments
  • Unique case files: {unique_files} PDF documents
  • Data completeness: 100% (all fields populated)

⚖️ LEGAL COVERAGE:
  • Areas of law: {unique_areas} distinct categories
  • Topics: {unique_topics} subtopics
  • Primary statutes: {unique_statutes} laws referenced
  • Most covered area: {df['area_of_law'].value_counts().index[0]} ({df['area_of_law'].value_counts().values[0]} rows)
  • Most common statute: {df['primary_statute'].value_counts().index[0]} ({df['primary_statute'].value_counts().values[0]} rows)

🏛️ COURTS REPRESENTED:
  • Singapore Court of Appeal (SGCA): {(df['court_level'] == 'SGCA').sum():,} cases
  • Singapore High Court (SGHC): {(df['court_level'] == 'SGHC').sum():,} cases
  • Singapore District Court (SGDC): {(df['court_level'] == 'SGDC').sum():,} cases
  • Singapore PA (SGPA): {(df['court_level'] == 'SGPA').sum():,} cases

📅 TEMPORAL COVERAGE:
  • Date range: {df['year'].min()} — {df['year'].max()}
  • Most recent year: {df[df['year'] == df['year'].max()].shape[0]} cases in {df['year'].max()}
  • Most active year: {year_counts.idxmax()} with {year_counts.max()} cases

📝 CATCHWORD ANALYSIS:
  • Average catchword length: {df['catchword_length'].mean():.0f} characters
  • Average words per catchword: {df['catchword_word_count'].mean():.1f} words
  • Most frequent legal term: '{word_freq[0][0]}' (appears {word_freq[0][1]} times)

💾 DATA DISTRIBUTION:
  • Average catchwords per case: {len(df) / unique_cases:.1f}
  • Cases per taxonomy entry: {len(df) / df['taxonomy_key'].nunique():.1f}
"""

print(summary_insights)


📊 SINGAPORE CRIMINAL LAW DATASET — KEY INSIGHTS

📁 DATASET COMPOSITION:
  • Total data points: 1,683 rows (multiple catchwords per case)
  • Unique cases: 876 criminal judgments
  • Unique case files: 876 PDF documents
  • Data completeness: 100% (all fields populated)

⚖️ LEGAL COVERAGE:
  • Areas of law: 22 distinct categories
  • Topics: 79 subtopics
  • Primary statutes: 114 laws referenced
  • Most covered area: Criminal Law (1148 rows)
  • Most common statute: Criminal Procedure Code 2010 (697 rows)

🏛️ COURTS REPRESENTED:
  • Singapore Court of Appeal (SGCA): 413 cases
  • Singapore High Court (SGHC): 1,267 cases
  • Singapore District Court (SGDC): 0 cases
  • Singapore PA (SGPA): 0 cases

📅 TEMPORAL COVERAGE:
  • Date range: 2015 — 2026
  • Most recent year: 69 cases in 2026
  • Most active year: 2023 with 203 cases

📝 CATCHWORD ANALYSIS:
  • Average catchword length: 64 characters
  • Average words per catchword: 9.8 words
  • Most frequent legal term: 'criminal' (appears 16